<a href="https://colab.research.google.com/github/Izabelladesign/171FinalProj/blob/BasicModel/BasicModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kagglehub scikit-learn seaborn

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# check if we are running on google colab or locally
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    os.system("pip install -q kagglehub")

    # BEFORE RUNNING: add your kaggle token to colab secrets
    # 1. click the KEY ICON on the left sidebar in colab
    # 2. click "Add new secret"
    # 3. Name: KAGGLE_TOKEN   Value: your KGAT_... token from kaggle.com -> Settings -> API
    from google.colab import userdata
    os.environ["KAGGLE_TOKEN"] = userdata.get("KAGGLE_TOKEN")
    print("Kaggle token loaded!")

import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG19
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

# settings for training
IMG_SIZE    = (224, 224)  # resize all images to 224x224
BATCH_SIZE  = 32          # how many images to process at once
EPOCHS      = 20          # max number of times to loop through the data
SUBSET_SIZE = 3000        # only use 3000 images to save time (set to None for all 9k)
SEED        = 42          # random seed so results are reproducible

random.seed(SEED)

# download the dataset from kaggle
print("Downloading dataset...")
DATA_DIR = kagglehub.dataset_download("aneeshdighe/corals-classification")
print("Downloaded to:", DATA_DIR)

# the dataset is inside a subfolder called "Bleached Corals and Healthy Corals Classification"
DATA_DIR = os.path.join(DATA_DIR, "Bleached Corals and Healthy Corals Classification")

# point to the correct train/valid/test folders
TRAIN_DIR = os.path.join(DATA_DIR, "Training")
VALID_DIR = os.path.join(DATA_DIR, "Validation")
TEST_DIR  = os.path.join(DATA_DIR, "Testing")

print("Train:", TRAIN_DIR)
print("Valid:", VALID_DIR)
print("Test: ", TEST_DIR)


# load images from folders and apply augmentation to the training set
# augmentation means we randomly flip/rotate/zoom images to make the model more robust
def build_generators(subset_size=None):
    # augmentation only on training data
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,       # normalize pixel values from 0-255 to 0-1
        horizontal_flip=True,    # randomly flip images left/right
        vertical_flip=True,      # randomly flip images up/down
        rotation_range=20,       # randomly rotate up to 20 degrees
        zoom_range=0.2,          # randomly zoom in/out
        width_shift_range=0.1,   # randomly shift image horizontally
        height_shift_range=0.1,  # randomly shift image vertically
    )

    # no augmentation for validation and test, just normalize
    val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

    train_gen = train_datagen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode="binary", seed=SEED,
    )
    valid_gen = val_test_datagen.flow_from_directory(
        VALID_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode="binary", seed=SEED, shuffle=False,
    )
    test_gen = val_test_datagen.flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode="binary", seed=SEED, shuffle=False,
    )

    print(f"Classes: {train_gen.class_indices}")
    print(f"Train: {train_gen.samples} | Valid: {valid_gen.samples} | Test: {test_gen.samples}")
    return train_gen, valid_gen, test_gen


# model 1: a simple CNN we build from scratch
# conv layers learn visual patterns (edges, colors, shapes)
# pooling layers shrink the image size to reduce computation
# dropout randomly turns off neurons during training to prevent overfitting
# the final layer outputs a single number between 0 and 1 (0 = bleached, 1 = healthy)
def build_custom_cnn():
    model = models.Sequential([
        # block 1 - learns basic features like edges and colors
        layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(*IMG_SIZE, 3)),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(2, 2),
        layers.BatchNormalization(),
        layers.Dropout(0.25),

        # block 2 - learns more complex patterns
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(2, 2),
        layers.BatchNormalization(),
        layers.Dropout(0.25),

        # block 3 - learns even higher level features
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(2, 2),
        layers.BatchNormalization(),
        layers.Dropout(0.25),

        # flatten turns the 2D feature maps into a 1D list
        layers.Flatten(),
        # dense layers make the final classification decision
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),  # outputs probability
    ], name="Custom_CNN")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",  # standard loss for yes/no classification
        metrics=["accuracy"],
    )
    return model


# model 2: VGG19 pretrained on imagenet (transfer learning)
# instead of training from scratch, we use a model that already knows how to
# detect shapes, textures, and patterns from millions of images
# we just replace the last layer to classify our two coral classes
def build_vgg19():
    base = VGG19(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
    base.trainable = False  # freeze the pretrained weights for now

    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),  # summarize the features
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ], name="VGG19_Finetune")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model, base


# after the head is trained, we unfreeze the last block of VGG19
# and train it at a very low learning rate so we dont overwrite the pretrained knowledge
def unfreeze_vgg19_top(model, base_model, from_layer="block5_conv1"):
    base_model.trainable = True
    trainable = False
    for layer in base_model.layers:
        if layer.name == from_layer:
            trainable = True
        layer.trainable = trainable
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # very small lr
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    print(f"Unfrozen VGG19 from {from_layer} onwards")


# callbacks are things that happen automatically during training
# early stopping stops training if the model stops improving
# reduce lr slows down learning if we hit a plateau
# model checkpoint saves the best version of the model
def get_callbacks(model_name):
    return [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
        ModelCheckpoint(f"{model_name}_best.keras", monitor="val_accuracy",
                        save_best_only=True, verbose=1),
    ]


def train_model(model, train_gen, valid_gen, model_name, subset_size=None):
    # if we are using a subset, limit how many batches per epoch
    steps = (subset_size // BATCH_SIZE) if subset_size else len(train_gen)
    return model.fit(
        train_gen, steps_per_epoch=steps, epochs=EPOCHS,
        validation_data=valid_gen,
        callbacks=get_callbacks(model_name), verbose=1,
    )


# run the model on the test set and print all our metrics
def evaluate_model(model, test_gen, model_name):
    print(f"\nEvaluating: {model_name}")
    test_gen.reset()
    y_pred_prob = model.predict(test_gen, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()  # convert probabilities to 0 or 1
    y_true = test_gen.classes  # the actual labels

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    class_names = list(test_gen.class_indices.keys())
    print(classification_report(y_true, y_pred, target_names=class_names))

    # plot confusion matrix to see where the model is making mistakes
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"Confusion Matrix - {model_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.savefig(f"{model_name}_confusion_matrix.png", dpi=150)
    plt.show()

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


# plot accuracy and loss over epochs to see how training went
def plot_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["accuracy"],     label="Train")
    axes[0].plot(history.history["val_accuracy"], label="Val")
    axes[0].set_title(f"{model_name} - Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history["loss"],     label="Train")
    axes[1].plot(history.history["val_loss"], label="Val")
    axes[1].set_title(f"{model_name} - Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{model_name}_training_history.png", dpi=150)
    plt.show()


# bar chart comparing both models side by side
def compare_models(results: dict):
    metrics     = ["accuracy", "precision", "recall", "f1"]
    model_names = list(results.keys())
    x, width    = np.arange(len(metrics)), 0.3

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, name in enumerate(model_names):
        ax.bar(x + i * width, [results[name][m] for m in metrics], width, label=name)

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Model Comparison")
    ax.legend()
    plt.tight_layout()
    plt.savefig("model_comparison.png", dpi=150)
    plt.show()


# run everything
tf.random.set_seed(SEED)
np.random.seed(SEED)

train_gen, valid_gen, test_gen = build_generators(SUBSET_SIZE)
results = {}

# train the custom CNN
print("\nTraining Model 1: Custom CNN")
cnn = build_custom_cnn()
cnn.summary()
hist_cnn = train_model(cnn, train_gen, valid_gen, "CustomCNN", SUBSET_SIZE)
plot_history(hist_cnn, "CustomCNN")
results["Custom CNN"] = evaluate_model(cnn, test_gen, "CustomCNN")

# train VGG19
print("\nTraining Model 2: VGG19")
vgg, vgg_base = build_vgg19()
vgg.summary()

# first train just the new classifier head we added
hist_vgg1 = train_model(vgg, train_gen, valid_gen, "VGG19_phase1", SUBSET_SIZE)

# then unfreeze the last block and fine tune the whole thing
print("\nFine-tuning VGG19...")
unfreeze_vgg19_top(vgg, vgg_base)
hist_vgg2 = train_model(vgg, train_gen, valid_gen, "VGG19_finetune", SUBSET_SIZE)

# combine both training phases into one history for plotting
combined = type("H", (), {"history": {
    k: hist_vgg1.history[k] + hist_vgg2.history[k]
    for k in hist_vgg1.history
}})()
plot_history(combined, "VGG19_full")
results["VGG19"] = evaluate_model(vgg, test_gen, "VGG19")

# show final comparison between both models
compare_models(results)

Kaggle token loaded!
Using Colab cache for faster access to the 'corals-classification' dataset.
Downloaded to: /kaggle/input/corals-classification
Train: /kaggle/input/corals-classification/Bleached Corals and Healthy Corals Classification/Training
Valid: /kaggle/input/corals-classification/Bleached Corals and Healthy Corals Classification/Validation
Test:  /kaggle/input/corals-classification/Bleached Corals and Healthy Corals Classification/Testing
Found 7384 images belonging to 2 classes.
Found 985 images belonging to 2 classes.
Found 923 images belonging to 2 classes.
Classes: {'bleached_corals': 0, 'healthy_corals': 1}
Train: 7384 | Valid: 985 | Test: 923

Training Model 1: Custom CNN


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "Custom_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 112, 112, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 56, 56, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,978,529 (99.10 MB)

 Trainable params: 25,978,081 (99.10 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 636ms/step - accuracy: 0.5777 - loss: 5.4430
Epoch 1: val_accuracy improved from None to 0.52893, saving model to CustomCNN_best.keras

Epoch 1: finished saving model to CustomCNN_best.keras
93/93 ━━━━━━━━━━━━━━━━━━━━ 93s 763ms/step - accuracy: 0.5876 - loss: 2.8557 - val_accuracy: 0.5289 - val_loss: 0.7099 - learning_rate: 0.0010
Epoch 2/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 454ms/step - accuracy: 0.6758 - loss: 0.6188
Epoch 2: val_accuracy did not improve from 0.52893
93/93 ━━━━━━━━━━━━━━━━━━━━ 44s 481ms/step - accuracy: 0.6704 - loss: 0.6229 - val_accuracy: 0.4558 - val_loss: 0.8394 - learning_rate: 0.0010
Epoch 3/20
45/93 ━━━━━━━━━━━━━━━━━━━━ 21s 452ms/step - accuracy: 0.7066 - loss: 0.5711

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 3: val_accuracy did not improve from 0.52893
93/93 ━━━━━━━━━━━━━━━━━━━━ 22s 237ms/step - accuracy: 0.7056 - loss: 0.5763 - val_accuracy: 0.4893 - val_loss: 0.8407 - learning_rate: 0.0010
Epoch 4/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 467ms/step - accuracy: 0.6644 - loss: 0.6345
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 4: val_accuracy did not improve from 0.52893
93/93 ━━━━━━━━━━━━━━━━━━━━ 46s 491ms/step - accuracy: 0.6808 - loss: 0.6047 - val_accuracy: 0.4904 - val_loss: 0.7683 - learning_rate: 0.0010
Epoch 5/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 437ms/step - accuracy: 0.7164 - loss: 0.5583
Epoch 5: val_accuracy improved from 0.52893 to 0.63858, saving model to CustomCNN_best.keras

Epoch 5: finished saving model to CustomCNN_best.keras
93/93 ━━━━━━━━━━━━━━━━━━━━ 50s 545ms/step - accuracy: 0.7227 - loss: 0.5560 - val_accuracy: 0.6386 - val_loss: 0.6321 - learning_rate: 5.0000e-04
Epoch 6/20
45/93 ━━━━━━━━━━━━━━━━━━━━ 19s 404ms/step - accuracy: 0.73